DATA TRANSFORMATION

In [ ]:
import pandas as pd
import numpy as np

# Read the original dataset
df = pd.read_csv("amazon_prime_titles.csv")

# Display first 5 rows
df.head()

CREATE A WORKING COPY

In [ ]:
prime = df.copy()
print("Shape:", prime.shape)

REMOVE UNNECESSARY COLUMNS

`show_id` is only a running ID and `description` is long free text, so both are not needed for the analysis.

In [ ]:
prime = prime.drop(["show_id", "description"], axis=1, errors="ignore")

prime.head()

In [ ]:
print(prime.columns)

CHECK FOR DUPLICATES

In [ ]:
print("Duplicate rows:", prime.duplicated().sum())
prime = prime.drop_duplicates()
print("Rows after removing duplicates:", prime.shape[0])

RENAME THE COLUMNS

In [ ]:
prime = prime.rename(columns={
    'type': 'content_type',
    'title': 'title',
    'director': 'director',
    'cast': 'cast',
    'country': 'country',
    'date_added': 'date_added',
    'release_year': 'release_year',
    'rating': 'age_rating',
    'duration': 'duration',
    'listed_in': 'genres'
})

prime.head()

CHANGE DATA TYPES

`astype()` converts a whole column, `pd.to_numeric()` and `pd.to_datetime()` convert text safely.

In [ ]:
prime['content_type'] = prime['content_type'].astype('category')
prime['release_year'] = prime['release_year'].astype(int)

# date_added is text like 'March 30, 2021' -> convert to a real date
prime['date_added'] = pd.to_datetime(prime['date_added'].str.strip(), format='%B %d, %Y', errors='coerce')

prime.dtypes

ADDING NEW COLUMNS

In [ ]:
# Split '113 min' / '2 Seasons' into a number and a unit
prime['duration_value'] = pd.to_numeric(prime['duration'].str.extract(r'(\d+)')[0], errors='coerce')
prime['duration_unit'] = prime['duration'].str.extract(r'([A-Za-z]+)')[0]

# Movies -> runtime in minutes, TV shows -> number of seasons (kept separate so units are never mixed)
prime['runtime_minutes'] = prime['duration_value'].where(prime['duration_unit'] == 'min')
prime['seasons'] = prime['duration_value'].where(prime['duration_unit'] != 'min')

prime[['title', 'duration', 'duration_value', 'duration_unit', 'runtime_minutes', 'seasons']].head(10)

In [ ]:
# Number of genres listed for each title (apply + lambda)
prime['genre_count'] = prime['genres'].apply(lambda x: len(x.split(',')))

prime[['title', 'genres', 'genre_count']].head(10)

In [ ]:
# Age of the title when it was added to the catalogue (2021 is the latest year in the data)
prime['title_age'] = 2021 - prime['release_year']

# Release decade
prime['decade'] = (prime['release_year'] // 10 * 10).astype(str) + 's'

prime[['title', 'release_year', 'title_age', 'decade']].head(10)

In [ ]:
# Group the age rating into audience categories (apply + lambda)
prime['audience'] = prime['age_rating'].apply(
    lambda r: 'Unrated' if pd.isna(r)
    else 'Family' if r in ['ALL', '7+', 'PG', 'G', 'TV-Y', 'TV-Y7', 'TV-G', 'TV-PG']
    else 'Adult' if r in ['18+', 'R', 'NC-17', 'TV-MA', 'AO']
    else 'Teen'
)

prime[['title', 'age_rating', 'audience']].head(10)

Handling Null Values

In [ ]:
print(prime.isnull().sum())

Numeric columns: mean and median

In [ ]:
release_mean = prime['release_year'].mean()
release_median = prime['release_year'].median()
runtime_mean = prime['runtime_minutes'].mean()
runtime_median = prime['runtime_minutes'].median()

print("Mean release year:", round(release_mean, 1))
print("Median release year:", release_median)
print("Mean movie runtime (min):", round(runtime_mean, 1))
print("Median movie runtime (min):", runtime_median)

In [ ]:
# Filling movie runtime with the median. Movies have no missing runtime here, so this is a safeguard;
# the empty runtime values for TV shows are expected and are NOT filled.
is_movie = prime['content_type'] == 'Movie'
prime.loc[is_movie, 'runtime_minutes'] = prime.loc[is_movie, 'runtime_minutes'].fillna(runtime_median)
prime['release_year'] = prime['release_year'].fillna(release_median)

Categorical columns: mode

In [ ]:
age_rating_mode = prime['age_rating'].mode()[0]
print("Most common age rating:", age_rating_mode)

prime['age_rating'] = prime['age_rating'].fillna(age_rating_mode)

In [ ]:
# Text columns: use an explicit label instead of guessing a value
prime['director'] = prime['director'].fillna('Unknown')
prime['cast'] = prime['cast'].fillna('Unknown')
prime['country'] = prime['country'].fillna('Unknown')

# date_added is empty for ~98% of rows, so it is left as NaT on purpose
print("date_added missing:", prime['date_added'].isnull().sum(), "of", len(prime))

In [ ]:
print(prime.isnull().sum())

In [ ]:
print("Rows after null handling:", prime.shape[0])

In [ ]:
prime[['title', 'age_rating', 'director', 'country']].head(10)

In [ ]:
df.isnull()

In [ ]:
print(prime.columns.tolist())

SAVE THE CLEANED DATA

In [ ]:
prime.to_csv("amazon_prime_cleaned.csv", index=False)